# Financial Anomaly Detection & Peer-Group Profiling
### An unsupervised machine-learning study using the earlier 100-company finance dataset

**Student:** Paras Satpute  
**Division / Roll:** A1-05  
**PRN:** 202401040027

---

## Project problem statement

Financial analysts often need to screen large company universes to identify businesses whose operating profiles differ materially from comparable firms. This project builds an **unsupervised financial screening workflow** that:

1. Groups companies into peer groups based on scale, growth and operating efficiency.
2. Detects companies with unusual multivariate financial profiles using multiple anomaly-detection algorithms.
3. Compares anomaly methods, studies parameter sensitivity, and explains why each flagged company is unusual.

**Important interpretation:** an anomaly is an observation that is unusual relative to this dataset. It is **not evidence of fraud, insolvency, poor management, or an investment signal**.

## Why this is a good fit for the AI/ML syllabus

This project directly applies Unit 6 topics: anomaly detection, partitional clustering, density-based clustering, dimensionality reduction and PCA. It also uses statistical summaries, visualization, preprocessing, model selection and critical evaluation.

## Dataset scope and limitations

The notebook embeds the exact 100-row, 7-column company snapshot used in the previous finance regression assignment. It is cross-sectional, not a multi-year panel. The data therefore support **peer profiling and unusual-profile screening**, not time-series forecasting or real-time market prediction. All conclusions are conditional on the companies and values present in this small snapshot.

In [ ]:
import base64, io, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
CSV_B64 = "UmFuayxDb21wYW55LEluZHVzdHJ5LFJldmVudWVfVVNEX01pbGxpb25zLFJldmVudWVfR3Jvd3RoX1BjdCxFbXBsb3llZXMsSGVhZHF1YXJ0ZXJzCjEsV2FsbWFydCxSZXRhaWwsNjgwOTg1LDUuMSwyMTAwMDAwLCJCZW50b252aWxsZSwgQXJrYW5zYXMiCjIsQW1hem9uLFJldGFpbCBhbmQgY2xvdWQgY29tcHV0aW5nLDYzNzk1OSwxMS4wLDE1NTYwMDAsIlNlYXR0bGUsIFdhc2hpbmd0b24iCjMsVW5pdGVkSGVhbHRoIEdyb3VwLEhlYWx0aGNhcmUsNDAwMjc4LDcuNyw0MDAwMDAsIk1pbm5ldG9ua2EsIE1pbm5lc290YSIKNCxBcHBsZSxUZWNobm9sb2d5LDM5MTAzNSwyLjAsMTY0MDAwLCJDdXBlcnRpbm8sIENhbGlmb3JuaWEiCjUsQ1ZTIEhlYWx0aCxIZWFsdGhjYXJlLDM3MjgwOSw0LjIsMjU5NTAwLCJXb29uc29ja2V0LCBSaG9kZSBJc2xhbmQiCjYsQmVya3NoaXJlIEhhdGhhd2F5LENvbmdsb21lcmF0ZSwzNzE0MzMsMS45LDM5MjQwMCwiT21haGEsIE5lYnJhc2thIgo3LEFscGhhYmV0LFRlY2hub2xvZ3kgYW5kIGNsb3VkIGNvbXB1dGluZywzNTAwMTgsMTMuOSwxODMzMjMsIk1vdW50YWluIFZpZXcsIENhbGlmb3JuaWEiCjgsRXh4b24gTW9iaWwsUGV0cm9sZXVtLDM0OTU5NSwxLjUsNjA5MDAsIlNwcmluZywgVGV4YXMiCjksTWNLZXNzb24gQ29ycG9yYXRpb24sSGVhbHRoY2FyZSwzMDg5NTEsMTEuNyw0ODAwMCwiSXJ2aW5nLCBUZXhhcyIKMTAsQ2VuY29yYSxQaGFybWFjeSB3aG9sZXNhbGUsMjkzOTU5LDEyLjEsNDQwMDAsIkNvbnNob2hvY2tlbiwgUGVubnN5bHZhbmlhIgoxMSxKUE1vcmdhbiBDaGFzZSxGaW5hbmNpYWxzLDI3ODkwNiwxNi41LDMxNzIzMywiTmV3IFlvcmsgQ2l0eSwgTmV3IFlvcmsiCjEyLENvc3RjbyBXaG9sZXNhbGUsUmV0YWlsLDI1NDQ1Myw1LjAsMzMzMDAwLCJJc3NhcXVhaCwgV2FzaGluZ3RvbiIKMTMsQ2lnbmEsSGVhbHRoIGluc3VyYW5jZSwyNDcxMjEsMjYuNiw3MjM5OCwiQmxvb21maWVsZCwgQ29ubmVjdGljdXQiCjE0LE1pY3Jvc29mdCxUZWNobm9sb2d5IGFuZCBjbG91ZCBjb21wdXRpbmcsMjQ1MTIyLDE1LjcsMjI4MDAwLCJSZWRtb25kLCBXYXNoaW5ndG9uIgoxNSxDYXJkaW5hbCBIZWFsdGgsSGVhbHRoY2FyZSwyMjY4MjcsMTAuNiw0ODQxMSwiRHVibGluLCBPaGlvIgoxNixDaGV2cm9uLFBldHJvbGV1bSwyMDI3OTIsMC45LDQ1Mjk4LCJIb3VzdG9uLCBUZXhhcyIKMTcsQmFuayBvZiBBbWVyaWNhLEZpbmFuY2lhbHMsMTkyNDM0LDExLjksMjEzMTkzLCJDaGFybG90dGUsIE5vcnRoIENhcm9saW5hIgoxOCxHZW5lcmFsIE1vdG9ycyxBdXRvbW90aXZlLDE4NzQ0Miw5LjEsMTYyMDAwLCJEZXRyb2l0LCBNaWNoaWdhbiIKMTksRm9yZCBNb3RvcixBdXRvbW90aXZlLDE4NDk5Miw1LjAsMTcxMDAwLCJEZWFyYm9ybiwgTWljaGlnYW4iCjIwLEVsZXZhbmNlIEhlYWx0aCxIZWFsdGhjYXJlLDE3NzAxMSwzLjMsMTAzNjc5LCJJbmRpYW5hcG9saXMsIEluZGlhbmEiCjIxLENpdGlncm91cCxGaW5hbmNpYWxzLDE3MDc1Nyw4LjksMjI3ODU1LCJOZXcgWW9yayBDaXR5LCBOZXcgWW9yayIKMjIsTWV0YSBQbGF0Zm9ybXMsVGVjaG5vbG9neSwxNjQ1MDEsMjEuOSw3NDA2NywiTWVubG8gUGFyaywgQ2FsaWZvcm5pYSIKMjMsQ2VudGVuZSxIZWFsdGhjYXJlLDE2MzA3MSw1LjksNjA1MDAsIlN0LiBMb3VpcywgTWlzc291cmkiCjI0LFRoZSBIb21lIERlcG90LFJldGFpbCwxNTk1MTQsNC41LDQ3MDEwMCwiQXRsYW50YSwgR2VvcmdpYSIKMjUsRmFubmllIE1hZSxGaW5hbmNpYWxzLDE1MjY3MCw4LjEsODIwMCwiV2FzaGluZ3RvbiwgRC5DLiIKMjYsV2FsZ3JlZW5zIEJvb3RzIEFsbGlhbmNlLFBoYXJtYWNldXRpY2FsLDE0NzY1OCw2LjIsMjUyNTAwLCJEZWVyZmllbGQsIElsbGlub2lzIgoyNyxLcm9nZXIsUmV0YWlsLDE0NzEyMywtMS45LDQwOTAwMCwiQ2luY2lubmF0aSwgT2hpbyIKMjgsUGhpbGxpcHMgNjYsUGV0cm9sZXVtLDE0NTQ5NiwtMi45LDEzMjAwLCJIb3VzdG9uLCBUZXhhcyIKMjksTWFyYXRob24gUGV0cm9sZXVtLFBldHJvbGV1bSwxNDA0MTIsLTYuNiwxODMwMCwiRmluZGxheSwgT2hpbyIKMzAsVmVyaXpvbiBDb21tdW5pY2F0aW9ucyxUZWxlY29tbXVuaWNhdGlvbnMsMTM0Nzg4LDAuNiw5OTYwMCwiTmV3IFlvcmsgQ2l0eSwgTmV3IFlvcmsiCjMxLE52aWRpYSxUZWNobm9sb2d5LDEzMDQ5NywxMTQuMiwzNjAwMCwiU2FudGEgQ2xhcmEsIENhbGlmb3JuaWEiCjMyLEdvbGRtYW4gU2FjaHMgR3JvdXAsRmluYW5jaWFscywxMjY4NTMsMTcuMCw0NjUwMCwiTmV3IFlvcmsgQ2l0eSwgTmV3IFlvcmsiCjMzLFdlbGxzIEZhcmdvLEZpbmFuY2lhbHMsMTI1Mzk3LDguNywyMTc1MDIsIlNhbiBGcmFuY2lzY28sIENhbGlmb3JuaWEiCjM0LFZhbGVybyBFbmVyZ3ksUGV0cm9sZXVtLDEyMzk3NCwtMTAuOCw5OTEwLCJTYW4gQW50b25pbywgVGV4YXMiCjM1LENvbWNhc3QsVGVsZWNvbW11bmljYXRpb25zLDEyMzczMSwxLjgsMTgyMDAwLCJQaGlsYWRlbHBoaWEsIFBlbm5zeWx2YW5pYSIKMzYsU3RhdGUgRmFybSBJbnN1cmFuY2UsRmluYW5jaWFscywxMjI5NTEsMTguMCw2NzM4MSwiQmxvb21pbmd0b24sIElsbGlub2lzIgozNyxBVCZULENvbmdsb21lcmF0ZSBhbmQgdGVsZWNvbW11bmljYXRpb25zLDEyMjMzNiwtMC4xLDE0MDk5MCwiRGFsbGFzLCBUZXhhcyIKMzgsRnJlZGRpZSBNYWMsRmluYW5jaWFscywxMjIwNTIsMTMuMCw4MDkwLCJNY0xlYW4sIFZpcmdpbmlhIgozOSxIdW1hbmEsSGVhbHRoIGluc3VyYW5jZSwxMTc3NjEsMTAuNyw2NTY4MCwiTG91aXN2aWxsZSwgS2VudHVja3kiCjQwLE1vcmdhbiBTdGFubGV5LEZpbmFuY2lhbHMsMTA3Mjg1LDExLjUsODA0NzgsIk5ldyBZb3JrIENpdHksIE5ldyBZb3JrIgo0MSxUYXJnZXQsUmV0YWlsLDEwNjU2NiwtMC44LDQ0MDAwMCwiTWlubmVhcG9saXMsIE1pbm5lc290YSIKNDIsU3RvbmVYIEdyb3VwLEZpbmFuY2lhbHMsOTk4ODgsNjQuMSw0NTAwLCJOZXcgWW9yayBDaXR5LCBOZXcgWW9yayIKNDMsVGVzbGEsQXV0b21vdGl2ZSBhbmQgZW5lcmd5LDk3NjkwLDAuOSwxMjU2NjUsIkF1c3RpbiwgVGV4YXMiCjQ0LERlbGwgVGVjaG5vbG9naWVzLFRlY2hub2xvZ3ksOTU1NjcsOC4xLDEwODAwMCwiUm91bmQgUm9jaywgVGV4YXMiCjQ1LFBlcHNpQ28sQmV2ZXJhZ2UsOTE4NTQsMC40LDMxOTAwMCwiUHVyY2hhc2UsIE5ldyBZb3JrIgo0NixXYWx0IERpc25leSxNZWRpYSw5MTM2MSwyLjgsMjA1MDQwLCJCdXJiYW5rLCBDYWxpZm9ybmlhIgo0NyxVbml0ZWQgUGFyY2VsIFNlcnZpY2UsVHJhbnNwb3J0YXRpb24sOTEwNzAsMC4xLDM3MjE4MCwiQXRsYW50YSwgR2VvcmdpYSIKNDgsSm9obnNvbiAmIEpvaG5zb24sUGhhcm1hY2V1dGljYWwsODg4MjEsLTYuNywxMzgxMDAsIk5ldyBCcnVuc3dpY2ssIE5ldyBKZXJzZXkiCjQ5LEZlZEV4LFRyYW5zcG9ydGF0aW9uLDg3NjkzLC0yLjcsNDIyMTAwLCJNZW1waGlzLCBUZW5uZXNzZWUiCjUwLEFyY2hlciBEYW5pZWxzIE1pZGxhbmQsRm9vZCBpbmR1c3RyeSw4NTUzMCwtOC45LDQzMjEzLCJDaGljYWdvLCBJbGxpbm9pcyIKNTEsUHJvY3RlciAmIEdhbWJsZSxDb25zdW1lciBwcm9kdWN0cyBtYW51ZmFjdHVyaW5nLDg0MDM5LDIuNSwxMDgwMDAsIkNpbmNpbm5hdGksIE9oaW8iCjUyLExvd2UncyxSZXRhaWwsODM2NzQsLTMuMSwyMTU1MDAsIk1vb3Jlc3ZpbGxlLCBOb3J0aCBDYXJvbGluYSIKNTMsRW5lcmd5IFRyYW5zZmVyIFBhcnRuZXJzLFBldHJvbGV1bSw4MjY3MSw1LjIsMTYyNDgsIkRhbGxhcywgVGV4YXMiCjU0LFJUWCxBZXJvc3BhY2UgYW5kIGRlZmVuc2UsODA3MzksMTcuMSwxODYwMDAsIkFybGluZ3RvbiBDb3VudHksIFZpcmdpbmlhIgo1NSxBbGJlcnRzb25zLFJldGFpbCw3OTIzOCwyLjAsMTk2NjUwLCJCb2lzZSwgSWRhaG8iCjU2LFN5c2NvLEZvb2QgcHJvY2Vzc2luZyw3ODg0NCwzLjMsNzYwMDAsIkhvdXN0b24sIFRleGFzIgo1NyxQcm9ncmVzc2l2ZSxJbnN1cmFuY2UsNzUzNzIsMjEuNCw2NjMwOCwiTWF5ZmllbGQgVmlsbGFnZSwgT2hpbyIKNTgsQW1lcmljYW4gRXhwcmVzcyxGaW5hbmNpYWxzLDc0MjAxLDEwLjEsNzUxMDAsIk5ldyBZb3JrIENpdHksIE5ldyBZb3JrIgo1OSxMb2NraGVlZCBNYXJ0aW4sQWVyb3NwYWNlIGFuZCBkZWZlbnNlLDcxMDQzLDUuMSwxMjEwMDAsIkJldGhlc2RhLCBNYXJ5bGFuZCIKNjAsTWV0TGlmZSxGaW5hbmNpYWxzLDcwOTg2LDYuMSw0NTAwMCwiTmV3IFlvcmsgQ2l0eSwgTmV3IFlvcmsiCjYxLEhDQSBIZWFsdGhjYXJlLEhlYWx0aGNhcmUsNzA2MDMsOC43LDI3MTAwMCwiTmFzaHZpbGxlLCBUZW5uZXNzZWUiCjYyLFBydWRlbnRpYWwgRmluYW5jaWFsLEZpbmFuY2lhbHMsNzA0MDUsMzAuNCwzNzkzNiwiTmV3YXJrLCBOZXcgSmVyc2V5Igo2MyxCb2VpbmcsQWVyb3NwYWNlIGFuZCBkZWZlbnNlLDY2NTE3LC0xNC41LDE3MjAwMCwiQXJsaW5ndG9uIENvdW50eSwgVmlyZ2luaWEiCjY0LENhdGVycGlsbGFyLE1hY2hpbmVyeSw2NDgwOSwtMy40LDExMjkwMCwiSXJ2aW5nLCBUZXhhcyIKNjUsTWVyY2ssUGhhcm1hY2V1dGljYWwgaW5kdXN0cnksNjQxNjgsNi43LDc0MDAwLCJLZW5pbHdvcnRoLCBOZXcgSmVyc2V5Igo2NixBbGxzdGF0ZSxJbnN1cmFuY2UsNjQxMDYsMTIuMyw1NTIwMCwiTm9ydGhmaWVsZCBUb3duc2hpcCwgSWxsaW5vaXMiCjY3LFBmaXplcixQaGFybWFjZXV0aWNhbCBpbmR1c3RyeSw2MzYyNyw4LjgsODEwMDAsIk5ldyBZb3JrIENpdHksIE5ldyBZb3JrIgo2OCxJQk0sVGVjaG5vbG9neSBhbmQgY2xvdWQgY29tcHV0aW5nLDYyNzUzLDEuNCwyODQ1MDAsIkFybW9uaywgTmV3IFlvcmsiCjY5LE5ldyBZb3JrIExpZmUgSW5zdXJhbmNlIENvbXBhbnksSW5zdXJhbmNlLDYyNjM5LDE1LjMsMTUxMzEsIk5ldyBZb3JrIENpdHksIE5ldyBZb3JrIgo3MCxEZWx0YSBBaXIgTGluZXMsQWlybGluZSw2MTY0Myw2LjIsMTAzMDAwLCJBdGxhbnRhLCBHZW9yZ2lhIgo3MSxQdWJsaXggU3VwZXIgTWFya2V0cyxSZXRhaWwsNjAxNzcsNC42LDI1NTAwMCwiTGFrZWxhbmQsIEZsb3JpZGEiCjcyLE5hdGlvbndpZGUgTXV0dWFsIEluc3VyYW5jZSBDb21wYW55LEZpbmFuY2lhbHMsNTg2NDYsNy40LDIyNDUzLCJDb2x1bWJ1cywgT2hpbyIKNzMsVEQgU3lubmV4LEluZm9ybWF0aW9uIFRlY2hub2xvZ3ksNTg0NTIsMS42LDI1NzUwLCJDbGVhcndhdGVyLCBGbG9yaWRhIgo3NCxVbml0ZWQgQWlybGluZXMgSG9sZGluZ3MsQWlybGluZSw1NzA2Myw2LjIsMTA3MzAwLCJDaGljYWdvLCBJbGxpbm9pcyIKNzUsQ29ub2NvUGhpbGxpcHMsUGV0cm9sZXVtLDU2OTUzLC0yLjgsMTE4MDAsIkhvdXN0b24sIFRleGFzIgo3NixUSlgsUmV0YWlsLDU2MzYwLDQuMCwzNjQwMDAsIkZyYW1pbmdoYW0sIE1hc3NhY2h1c2V0dHMiCjc3LEFiYlZpZSxQaGFybWFjZXV0aWNhbCw1NjMzNCwzLjcsNTUwMDAsIkxha2UgQmx1ZmYsIElsbGlub2lzIgo3OCxFbnRlcnByaXNlIFByb2R1Y3RzIFBhcnRuZXJzLFBldHJvbGV1bSw1NjIxOSwxMy4xLDc4MDAsIkhvdXN0b24sIFRleGFzIgo3OSxDaGFydGVyIENvbW11bmljYXRpb25zLFRlbGVjb21tdW5pY2F0aW9ucyw1NTA4NSwwLjksOTQ1MDAsIlN0YW1mb3JkLCBDb25uZWN0aWN1dCIKODAsUGVyZm9ybWFuY2UgRm9vZCBHcm91cCxGb29kIHByb2Nlc3NpbmcsNTQ2ODEsMi41LDM2ODE1LCJSaWNobW9uZCwgVmlyZ2luaWEiCjgxLEFtZXJpY2FuIEFpcmxpbmVzIEdyb3VwLEFpcmxpbmUsNTQyMTEsMi43LDEzMzMwMCwiRm9ydCBXb3J0aCwgVGV4YXMiCjgyLENhcGl0YWwgT25lIEZpbmFuY2lhbCxGaW5hbmNpYWxzLDUzOTM4LDkuMCw1MjYwMCwiVHlzb25zLCBWaXJnaW5pYSIKODMsQ2lzY28gU3lzdGVtcyxUZWxlY29tIGhhcmR3YXJlIG1hbnVmYWN0dXJpbmcsNTM4MDMsLTUuNiw5MDQwMCwiU2FuIEpvc2UsIENhbGlmb3JuaWEiCjg0LEhQLFRlY2hub2xvZ3ksNTM1NTksLTAuMyw1ODAwMCwiUGFsbyBBbHRvLCBDYWxpZm9ybmlhIgo4NSxUeXNvbiBGb29kcyxGb29kIHByb2Nlc3NpbmcsNTMzMDksMC44LDEzODAwMCwiU3ByaW5nZGFsZSwgQXJrYW5zYXMiCjg2LEludGVsLFRlY2hub2xvZ3ksNTMxMDEsLTIuMSwxMDg5MDAsIlNhbnRhIENsYXJhLCBDYWxpZm9ybmlhIgo4NyxPcmFjbGUsVGVjaG5vbG9neSw1Mjk2MSw2LjAsMTU5MDAwLCJBdXN0aW4sIFRleGFzIgo4OCxCcm9hZGNvbSxUZWNobm9sb2d5LDUyNDMyLDQ2LjQsMzcwMDAsIlBhbG8gQWx0bywgQ2FsaWZvcm5pYSIKODksRGVlcmUsQWdyaWN1bHR1cmUgbWFudWZhY3R1cmluZyw1MTcxNiwtMTUuNiw1NTUyNCwiTW9saW5lLCBJbGxpbm9pcyIKOTAsTmlrZSxBcHBhcmVsLDUxMzYxLDAuMyw3ODQwMCwiQmVhdmVydG9uLCBPcmVnb24iCjkxLExpYmVydHkgTXV0dWFsIEluc3VyYW5jZSBHcm91cCxJbnN1cmFuY2UsNTA0MjksLTQuMSw0MDAwMCwiQm9zdG9uLCBNYXNzYWNodXNldHRzIgo5MixQbGFpbnMgR1AgSG9sZGluZ3MsUGV0cm9sZXVtLDUwMDczLDIuOCw0MjAwLCJIb3VzdG9uLCBUZXhhcyIKOTMsVVNBQSxJbnN1cmFuY2UsNDg1NjAsMTQuMywzODAxOCwiU2FuIEFudG9uaW8sIFRleGFzIgo5NCxCcmlzdG9sLU15ZXJzIFNxdWliYixQaGFybWFjZXV0aWNhbCw0ODMwMCw3LjMsMzQxMDAsIk5ldyBZb3JrLCBOZXcgWW9yayIKOTUsSW5ncmFtIE1pY3JvIEhvbGRpbmcsVGVjaG5vbG9neSw0Nzk4NCwtMC4xLDI2MTI1LCJJcnZpbmUsIENhbGlmb3JuaWEiCjk2LEdlbmVyYWwgRHluYW1pY3MsQWVyb3NwYWNlIGFuZCBkZWZlbnNlLDQ3NzE2LDEyLjksMTE3MDAwLCJSZXN0b24sIFZpcmdpbmlhIgo5NyxDb2NhLUNvbGEsQmV2ZXJhZ2UsNDcwNjEsMi45LDY5NzAwLCJBdGxhbnRhLCBHZW9yZ2lhIgo5OCxUSUFBLEZpbmFuY2lhbHMsNDY5NDYsMi42LDE1NjIzLCJOZXcgWW9yayBDaXR5LCBOZXcgWW9yayIKOTksVHJhdmVsZXJzLEluc3VyYW5jZSw0NjQyMywxMi4yLDM0MDAwLCJOZXcgWW9yayBDaXR5LCBOZXcgWW9yayIKMTAwLEVsaSBMaWxseSxQaGFybWFjZXV0aWNhbCw0NTA0MywzMi4wLDQ3MDAwLCJJbmRpYW5hcG9saXMsIEluZGlhbmEiCg=="
df = pd.read_csv(io.BytesIO(base64.b64decode(CSV_B64)))
print("Dataset shape:", df.shape)
display(df.head())

## 1. Data collection and attribute description

The input is the previously collected finance dataset from the earlier semester-log exercise. The notebook does not add synthetic rows or download replacement data.

| Attribute | Meaning | Role in this project |
|---|---|---|
| `Rank` | Original ordering/rank in the collected snapshot | Identifier/context only |
| `Company` | Company name | Identifier and output label |
| `Industry` | Industry classification | Context for interpreting peer groups |
| `Revenue_USD_Millions` | Revenue in USD millions | Financial scale feature |
| `Revenue_Growth_Pct` | Revenue growth percentage | Growth feature |
| `Employees` | Number of employees | Operating scale feature |
| `Headquarters` | Headquarters location | Context only; excluded from numerical distance |

We deliberately exclude `Rank`, `Company`, and `Headquarters` from model inputs: rank is an ordering field, company is an identifier, and headquarters is not a comparable operating metric. `Industry` is retained for interpretation rather than used as a one-hot distance feature; the small sample and many sector labels could make distances dominated by sparse category indicators.

In [ ]:
print("Rows, columns:", df.shape)
print("\nData types:")
display(df.dtypes.rename("dtype").to_frame())
print("\nMissing values:")
display(df.isna().sum().rename("missing").to_frame())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nUnique companies:", df["Company"].nunique())
print("\nIndustry counts:")
display(df["Industry"].value_counts().head(15).to_frame("companies"))

## 2. Statistical summary and initial observations

We inspect central tendency, spread, quantiles, skewness and industry counts. Financial scale variables are usually right-skewed: a small number of very large companies can dominate raw Euclidean distances. This motivates log transformation and robust scaling before distance-based methods.

In [ ]:
numeric_cols = ["Revenue_USD_Millions", "Revenue_Growth_Pct", "Employees"]
summary = df[numeric_cols].describe().T
summary["median"] = df[numeric_cols].median()
summary["skewness"] = df[numeric_cols].skew()
display(summary.round(3))
print("Revenue per employee (USD millions per employee) summary:")
rev_per_emp = df["Revenue_USD_Millions"] * 1_000_000 / df["Employees"]
display(rev_per_emp.describe().to_frame("Revenue_USD_per_employee").round(2))
print("Industry count:", df["Industry"].nunique())

### Statistical inferences to discuss

- Compare mean and median to see whether company scale is dominated by a few very large firms.
- Large positive skew in revenue or employee count justifies using `log1p` for scale variables.
- Revenue growth can be negative or highly positive; it is retained as a signed feature rather than logged.
- Revenue per employee is an interpretable productivity/operating-intensity proxy, but it varies substantially by industry. It must not be interpreted as a universal measure of managerial quality.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
ax[0,0].hist(df["Revenue_USD_Millions"], bins=18, edgecolor="black")
ax[0,0].set_title("Revenue (raw scale)")
ax[0,0].set_xlabel("USD millions")
ax[0,1].hist(np.log1p(df["Revenue_USD_Millions"]), bins=18, edgecolor="black")
ax[0,1].set_title("Log-transformed revenue")
ax[0,1].set_xlabel("log(1 + revenue)")
ax[1,0].hist(df["Revenue_Growth_Pct"], bins=18, edgecolor="black")
ax[1,0].set_title("Revenue growth distribution")
ax[1,0].set_xlabel("Growth (%)")
ax[1,1].hist(np.log1p(df["Employees"]), bins=18, edgecolor="black")
ax[1,1].set_title("Log-transformed employee count")
ax[1,1].set_xlabel("log(1 + employees)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
top_industries = df["Industry"].value_counts().head(12).index
df[df["Industry"].isin(top_industries)]["Industry"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Company counts in the 12 most represented industry labels")
ax.set_xlabel("Industry")
ax.set_ylabel("Number of companies")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8,5))
scatter = ax.scatter(df["Revenue_USD_Millions"], df["Employees"],
                     c=df["Revenue_Growth_Pct"], cmap="coolwarm", s=55, alpha=.8)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Revenue (USD millions, log axis)")
ax.set_ylabel("Employees (log axis)")
ax.set_title("Company scale profile (colour = revenue growth)")
plt.colorbar(scatter, ax=ax, label="Revenue growth (%)")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(df["Revenue_Growth_Pct"], rev_per_emp, alpha=.8)
for _, r in df.nlargest(5, "Revenue_Growth_Pct").iterrows():
    ax.annotate(r["Company"], (r["Revenue_Growth_Pct"],
                 r["Revenue_USD_Millions"]*1_000_000/r["Employees"]),
                 fontsize=8, xytext=(4,4), textcoords="offset points")
ax.set_xlabel("Revenue growth (%)")
ax.set_ylabel("Revenue per employee (USD)")
ax.set_title("Growth versus revenue per employee")
plt.tight_layout(); plt.show()

### Visual exploration inferences

The four visualizations above cover distributions (raw and transformed), industry representation, company scale, and growth versus revenue per employee. Use them to identify skew, sector imbalance, broad scale groups, and potentially unusual combinations. A visually extreme point is only a candidate: the multivariate models below assess unusualness across several features simultaneously.

## 3. Data preparation and feature engineering

### Design decisions

1. Validate numeric columns and handle missing/infinite values.
2. Derive `Revenue_per_Employee_USD` from revenue and employee count. This adds an interpretable efficiency dimension.
3. Apply `log1p` to revenue and employee count to compress the long right tail.
4. Keep growth signed (do not log it).
5. Use `RobustScaler`, which centers by the median and scales by the interquartile range, reducing sensitivity to extreme values.
6. Exclude identifiers and location from model features.

The model matrix uses four dimensions: log revenue, log employees, revenue growth, and log revenue per employee. The ratio is a descriptive proxy, not a sector-adjusted productivity score.

In [ ]:
work = df.copy()
for col in ["Revenue_USD_Millions", "Revenue_Growth_Pct", "Employees"]:
    work[col] = pd.to_numeric(work[col], errors="coerce")
work = work.replace([np.inf, -np.inf], np.nan)
work = work.dropna(subset=["Company", "Revenue_USD_Millions", "Revenue_Growth_Pct", "Employees"]).copy()
work = work[(work["Revenue_USD_Millions"] >= 0) & (work["Employees"] > 0)].copy()
work["Revenue_per_Employee_USD"] = (
    work["Revenue_USD_Millions"] * 1_000_000 / work["Employees"]
)
work["Log_Revenue"] = np.log1p(work["Revenue_USD_Millions"])
work["Log_Employees"] = np.log1p(work["Employees"])
work["Log_Revenue_per_Employee"] = np.log1p(work["Revenue_per_Employee_USD"])
feature_cols = ["Log_Revenue", "Log_Employees",
                "Revenue_Growth_Pct", "Log_Revenue_per_Employee"]
X_raw = work[feature_cols].copy()
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_raw)
X = pd.DataFrame(X_scaled, columns=feature_cols, index=work.index)
print("Rows retained after cleaning:", len(work))
print("Feature matrix shape:", X.shape)
display(work[["Company","Industry","Revenue_USD_Millions","Revenue_Growth_Pct",
              "Employees","Revenue_per_Employee_USD"]].head().round(2))

## 4. PCA: dimensionality reduction and visualization

PCA is used to project the four-dimensional robust-scaled feature space into two principal components for visualization. PCA is **not** used to define the only anomaly criteria; the primary algorithms operate on the prepared feature matrix. The explained-variance ratio indicates how much information the two-dimensional plot retains.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))
print("Cumulative variance in 2 PCs:", round(pca.explained_variance_ratio_.sum(), 3))
fig, ax = plt.subplots(figsize=(9,6))
ax.scatter(X_pca[:,0], X_pca[:,1], alpha=.75)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("PCA projection of company financial profiles")
plt.tight_layout(); plt.show()

## 5. Peer-group discovery with K-Means (partitional clustering)

K-Means is used to create broad peer groups based on the prepared financial profile. We compare `k = 2...6` using inertia (elbow diagnostic) and silhouette score. Silhouette is an internal clustering metric, not ground-truth accuracy. The final k is selected using both the score and interpretability; a small dataset does not justify excessively granular clusters.

In [ ]:
k_rows = []
kmeans_models = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = km.fit_predict(X)
    kmeans_models[k] = km
    k_rows.append({"k": k, "inertia": km.inertia_,
                   "silhouette": silhouette_score(X, labels)})
k_eval = pd.DataFrame(k_rows)
display(k_eval.round(3))
fig, ax1 = plt.subplots(figsize=(9,5))
ax1.plot(k_eval["k"], k_eval["inertia"], marker="o")
ax1.set_xlabel("Number of clusters (k)")
ax1.set_ylabel("Inertia")
ax1.set_title("K-Means model selection: elbow and silhouette")
ax2 = ax1.twinx()
ax2.plot(k_eval["k"], k_eval["silhouette"], marker="s", linestyle="--")
ax2.set_ylabel("Silhouette score")
plt.tight_layout(); plt.show()

In [ ]:
# Select k with the strongest silhouette among the tested values.
# The selected value is then checked against the cluster profiles below.
best_k = int(k_eval.loc[k_eval["silhouette"].idxmax(), "k"])
kmeans = kmeans_models[best_k]
work["Peer_Group"] = kmeans.labels_
print("Selected k:", best_k)
peer_profile = work.groupby("Peer_Group").agg(
    Companies=("Company","count"),
    Median_Revenue_M=("Revenue_USD_Millions","median"),
    Median_Employees=("Employees","median"),
    Median_Growth_Pct=("Revenue_Growth_Pct","median"),
    Median_Revenue_per_Employee_USD=("Revenue_per_Employee_USD","median")
).round(2)
display(peer_profile)
display(work.groupby("Peer_Group")["Industry"].agg(lambda s: s.value_counts().head(3).to_dict()).to_frame("Top industry labels"))

### Important interpretation of the K-Means result

In this dataset, the highest silhouette score is obtained at **k = 2**, but the resulting groups are highly imbalanced (97 companies versus 3). This is a warning that the score is being driven by a very distinct small group rather than discovering several balanced industry peer groups. We therefore interpret K-Means as a **coarse separation of the dominant company population from a small extreme-profile group**, not as a definitive industry peer taxonomy. The anomaly-detection methods below provide the main result of the project. This is an example of critically analysing an internal metric instead of blindly selecting the numerical maximum.

In [ ]:
fig, ax = plt.subplots(figsize=(9,6))
for g in sorted(work["Peer_Group"].unique()):
    mask = work["Peer_Group"].to_numpy() == g
    ax.scatter(X_pca[mask,0], X_pca[mask,1], label=f"Peer group {g}", alpha=.8)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("K-Means peer groups in PCA space")
ax.legend()
plt.tight_layout(); plt.show()

## 6. Anomaly detection: three complementary approaches

We compare algorithms that define unusualness differently:

- **Isolation Forest:** isolates observations using random feature splits; points isolated in fewer splits receive higher anomaly scores. It is suitable for multivariate tabular screening.
- **Local Outlier Factor (LOF):** compares each point's local density with that of its neighbours. It can identify a company that is unusual relative to nearby peers even if it is not globally extreme.
- **DBSCAN:** groups dense regions and labels points outside dense regions as noise (`-1`). Its results depend strongly on `eps` and `min_samples`; noise is not automatically a financial anomaly.

These methods are complementary rather than interchangeable. We compare flag counts, overlap, and sensitivity instead of claiming a supervised accuracy score when no verified anomaly labels exist.

In [ ]:
# Isolation Forest parameter sensitivity: contamination is the expected outlier fraction.
if_rows = []
if_models = {}
for contamination in [0.05, 0.08, 0.10, 0.15]:
    model = IsolationForest(
        n_estimators=300, contamination=contamination,
        random_state=RANDOM_STATE, max_samples="auto"
    )
    pred = model.fit_predict(X)
    score = -model.score_samples(X)  # larger = more anomalous
    if_models[contamination] = (model, pred, score)
    if_rows.append({"contamination": contamination,
                    "flagged": int((pred == -1).sum()),
                    "mean_flag_score_flagged": float(score[pred == -1].mean())})
display(pd.DataFrame(if_rows).round(4))

In [ ]:
# LOF sensitivity over neighbour counts; novelty=False is correct for in-sample unsupervised scoring.
lof_rows = []
lof_models = {}
for n_neighbors in [8, 10, 15, 20]:
    n_neighbors = min(n_neighbors, len(X)-1)
    model = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=0.10)
    pred = model.fit_predict(X)
    score = -model.negative_outlier_factor_
    lof_models[n_neighbors] = (model, pred, score)
    lof_rows.append({"n_neighbors": n_neighbors,
                     "flagged": int((pred == -1).sum()),
                     "mean_flag_score_flagged": float(score[pred == -1].mean())})
display(pd.DataFrame(lof_rows).round(4))

In [ ]:
# DBSCAN eps/min_samples sensitivity. Noise points are labelled -1.
db_rows = []
db_models = {}
for eps in [0.45, 0.60, 0.75, 0.90, 1.10, 1.30]:
    for min_samples in [3, 5]:
        model = DBSCAN(eps=eps, min_samples=min_samples)
        labels = model.fit_predict(X)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = int((labels == -1).sum())
        non_noise = labels != -1
        sil = np.nan
        if n_clusters >= 2 and non_noise.sum() > n_clusters:
            try:
                sil = silhouette_score(X[non_noise], labels[non_noise])
            except Exception:
                pass
        db_models[(eps,min_samples)] = (model, labels)
        db_rows.append({"eps":eps,"min_samples":min_samples,
                        "clusters":n_clusters,"noise_points":n_noise,
                        "silhouette_non_noise":sil})
db_eval = pd.DataFrame(db_rows)
display(db_eval.round(3))

### Selecting a defensible working configuration

We use a moderate 10% contamination for Isolation Forest and LOF as a screening setting, not as a known truth about the number of anomalies. For DBSCAN, inspect the sensitivity table and select a setting that produces interpretable dense groups without labelling nearly everything as noise. If no DBSCAN setting produces meaningful clusters, report that density clustering is not well suited to this dataset's geometry rather than forcing a result.

In [ ]:
IF_CONTAMINATION = 0.10
LOF_NEIGHBORS = min(15, len(X)-1)
if_model, if_pred, if_score = if_models[IF_CONTAMINATION]
lof_model, lof_pred, lof_score = lof_models[LOF_NEIGHBORS]
# Select DBSCAN setting with >=2 clusters and the largest non-noise silhouette,
# while requiring at least 70% of records to remain in dense clusters.
eligible = db_eval[(db_eval["clusters"] >= 2) &
                   (db_eval["noise_points"] <= 0.30*len(X)) &
                   (db_eval["silhouette_non_noise"].notna())]
if len(eligible):
    chosen = eligible.sort_values("silhouette_non_noise", ascending=False).iloc[0]
    db_key = (float(chosen["eps"]), int(chosen["min_samples"]))
else:
    # Explicit fallback: smallest-noise configuration; discuss if it yields one cluster.
    chosen = db_eval.sort_values("noise_points").iloc[0]
    db_key = (float(chosen["eps"]), int(chosen["min_samples"]))
db_model, db_labels = db_models[db_key]
db_pred = np.where(db_labels == -1, -1, 1)
print("Selected DBSCAN eps/min_samples:", db_key)
print("DBSCAN clusters:", len(set(db_labels)) - (1 if -1 in db_labels else 0),
      "| noise points:", int((db_labels == -1).sum()))

In [ ]:
# Combine model decisions. Each algorithm flags -1 as unusual.
comparison = pd.DataFrame({
    "Company": work["Company"].to_numpy(),
    "Industry": work["Industry"].to_numpy(),
    "Isolation_Forest": (if_pred == -1),
    "LOF": (lof_pred == -1),
    "DBSCAN_Noise": (db_labels == -1)
}, index=work.index)
comparison["Votes"] = comparison[["Isolation_Forest","LOF","DBSCAN_Noise"]].sum(axis=1)
comparison["Consensus_Flag"] = comparison["Votes"] >= 2
display(comparison[["Isolation_Forest","LOF","DBSCAN_Noise"]].sum().rename("flagged_companies").to_frame())
print("Companies flagged by at least two methods:", int(comparison["Consensus_Flag"].sum()))
print("Companies flagged by all three:", int((comparison["Votes"] == 3).sum()))

In [ ]:
# Algorithm overlap: Jaccard similarity of the sets flagged by each method.
flag_sets = {
    "Isolation Forest": set(work.loc[if_pred == -1, "Company"]),
    "LOF": set(work.loc[lof_pred == -1, "Company"]),
    "DBSCAN": set(work.loc[db_labels == -1, "Company"])
}
names = list(flag_sets)
jaccard = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        union = flag_sets[a] | flag_sets[b]
        jaccard.loc[a,b] = len(flag_sets[a] & flag_sets[b]) / len(union) if union else 1.0
display(jaccard.round(3))
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(jaccard.astype(float), vmin=0, vmax=1)
ax.set_xticks(range(len(names)), names, rotation=25, ha="right")
ax.set_yticks(range(len(names)), names)
ax.set_title("Jaccard overlap of anomaly flags")
plt.colorbar(im, ax=ax, label="Jaccard similarity")
plt.tight_layout(); plt.show()

## 7. Unsupervised validation and stability analysis

There are no verified anomaly labels in the dataset. Therefore, classification accuracy, precision, recall, F1 and ROC-AUC would be fabricated or invalid. Instead, this project uses:

- Internal clustering quality (silhouette) for peer groups.
- Parameter sensitivity (how flag counts change across contamination, neighbour count and DBSCAN density settings).
- Pairwise Jaccard overlap between methods.
- Subsample stability: refit Isolation Forest on repeated 80% subsamples and compare its top-ranked unusual companies with the full-data top-ranked set.

This is the unsupervised equivalent of checking whether findings persist under resampling. It is not a substitute for external ground-truth validation.

In [ ]:
# Repeated 80% subsampling stability for Isolation Forest top-10 anomaly ranking.
rng = np.random.default_rng(RANDOM_STATE)
full_top10 = set(work.iloc[np.argsort(if_score)[-10:]]["Company"])
stability_rows = []
for repeat in range(20):
    sample_pos = rng.choice(len(X), size=int(0.80*len(X)), replace=False)
    X_sub = X.iloc[sample_pos]
    sub_model = IsolationForest(n_estimators=250, contamination=0.10,
                                random_state=RANDOM_STATE+repeat)
    sub_model.fit(X_sub)
    sub_scores = -sub_model.score_samples(X_sub)
    top_local = np.argsort(sub_scores)[-min(10,len(sample_pos)):]
    sub_top = set(work.iloc[sample_pos[top_local]]["Company"])
    union = full_top10 | sub_top
    j = len(full_top10 & sub_top)/len(union) if union else 1.0
    stability_rows.append({"repeat":repeat+1,"top10_jaccard_vs_full":j})
stability = pd.DataFrame(stability_rows)
display(stability.describe().round(3))
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(stability["repeat"], stability["top10_jaccard_vs_full"], marker="o")
ax.set_ylim(0,1); ax.set_xlabel("80% subsample repeat")
ax.set_ylabel("Jaccard overlap with full-data top 10")
ax.set_title("Isolation Forest ranking stability")
plt.tight_layout(); plt.show()

## 8. Interpreting the unusual companies

A useful screening result should explain *why* a company was flagged. We rank observations by Isolation Forest anomaly score, show which other algorithms agree, and display the underlying financial attributes and peer-group assignment. Do not label the highest score as a confirmed fraud or failure.

In [ ]:
results = work[["Company","Industry","Revenue_USD_Millions","Revenue_Growth_Pct",
                "Employees","Revenue_per_Employee_USD","Peer_Group"]].copy()
results["IF_Anomaly_Score"] = if_score
results["IF_Flag"] = if_pred == -1
results["LOF_Flag"] = lof_pred == -1
results["DBSCAN_Noise"] = db_labels == -1
results["Consensus_Votes"] = comparison["Votes"].to_numpy()
results["Consensus_Flag"] = comparison["Consensus_Flag"].to_numpy()
ranked = results.sort_values("IF_Anomaly_Score", ascending=False)
display(ranked.head(15).round(3))

In [ ]:
# Show consensus candidates separately; if empty, use the top Isolation Forest candidates.
consensus_table = results[results["Consensus_Flag"]].sort_values(
    "IF_Anomaly_Score", ascending=False)
if consensus_table.empty:
    print("No 2-of-3 consensus cases. Showing top 10 Isolation Forest candidates instead.")
    consensus_table = ranked.head(10)
display(consensus_table[["Company","Industry","Revenue_USD_Millions",
                         "Revenue_Growth_Pct","Employees",
                         "Revenue_per_Employee_USD","Peer_Group",
                         "IF_Anomaly_Score","Consensus_Votes"]].round(2))

In [ ]:
# Visualize Isolation Forest flags and peer groups in PCA space.
fig, ax = plt.subplots(figsize=(10,6))
normal = if_pred != -1
ax.scatter(X_pca[normal,0], X_pca[normal,1], label="Not flagged by IF", alpha=.65)
ax.scatter(X_pca[~normal,0], X_pca[~normal,1], label="IF anomaly candidate",
           marker="X", s=95)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Isolation Forest anomaly candidates in PCA space")
ax.legend()
plt.tight_layout(); plt.show()

## 9. Algorithm comparison and critical analysis

| Method | What it detects | Strength | Limitation in this dataset |
|---|---|---|---|
| Isolation Forest | Globally isolatable points | Handles multivariate data; scalable | Contamination must be chosen; score is relative |
| LOF | Locally low-density points | Detects local peer deviations | Sensitive to neighbour count and scaling |
| DBSCAN | Points outside dense regions | No need to predefine number of clusters | Sensitive to `eps`; variable-density data can be difficult |
| K-Means | Broad peer groups | Simple, interpretable segment profiles | Assumes compact clusters; sensitive to scale and k |
| PCA | Main directions of variation | Makes multivariate structure visible | 2D projection loses information |

### What counts as a good result?

A good result is not simply the algorithm that flags the most companies. A defensible result has: (1) a plausible number of candidates, (2) some agreement across complementary methods, (3) stability under parameter/resampling checks, and (4) an interpretable financial profile. If methods disagree, that disagreement is itself an important result about the data's geometry.

In [ ]:
print("FINAL PROJECT SUMMARY")
print("="*60)
print(f"Companies analysed: {len(work)}")
print(f"Peer groups (K-Means): {best_k}")
print(f"K-Means silhouette: {k_eval.loc[k_eval.k==best_k,'silhouette'].iloc[0]:.3f}")
print(f"Isolation Forest candidates: {(if_pred==-1).sum()}")
print(f"LOF candidates: {(lof_pred==-1).sum()}")
print(f"DBSCAN noise points: {(db_labels==-1).sum()}")
print(f"2-of-3 consensus candidates: {comparison['Consensus_Flag'].sum()}")
print(f"Mean subsample top-10 Jaccard stability: {stability['top10_jaccard_vs_full'].mean():.3f}")
print("\nTop 5 Isolation Forest candidates:")
display(ranked[["Company","Industry","Revenue_Growth_Pct",
                "Revenue_per_Employee_USD","IF_Anomaly_Score",
                "Consensus_Votes"]].head(5).round(2))

## 10. Limitations, ethics and future improvements

1. **Small sample:** 100 companies are insufficient to learn a universal definition of financial normality.
2. **Cross-sectional snapshot:** no company-year history exists, so the project cannot identify changes over time or predict future distress.
3. **Sector effects:** revenue per employee and growth norms differ by industry. A sector-specific model would be preferable with a larger dataset.
4. **No verified labels:** anomaly flags are candidates for review, not measured fraud-detection accuracy.
5. **Feature coverage:** the dataset lacks debt, cash flow, margins, assets, market capitalization and accounting quality.
6. **Bias and representation:** industries and company sizes are not evenly represented; the dataset's collection method may introduce selection bias.
7. **Ethical use:** do not use anomaly flags alone to deny credit, rank employees, accuse companies, or make investment decisions. A human analyst should inspect source filings and context.

### Future improvements

- Collect several years of audited financial statements for each company.
- Add margins, debt ratios, cash-flow ratios and industry-normalized indicators.
- Build sector-specific peer groups and compare within-sector anomaly scores.
- Validate candidate anomalies with documented analyst review or external labels.
- Use time-aware monitoring once multiple years are available.

## 11. Conclusion

This mini-project turns the earlier finance dataset into an unsupervised financial screening study. K-Means discovers broad peer groups, PCA visualizes multivariate structure, and Isolation Forest, LOF and DBSCAN provide complementary anomaly perspectives. Parameter sensitivity, Jaccard overlap and subsample stability are used instead of misleading supervised accuracy metrics because the dataset has no verified anomaly labels.

The strongest conclusion is methodological: **unsupervised anomaly detection can prioritize unusual company profiles for further investigation, but it cannot explain the cause of unusualness or establish wrongdoing.** The results should be treated as exploratory evidence, with sector effects and the small cross-sectional sample kept in view.

---

## Reproducibility

All transformations, model seeds and parameter grids are specified in the notebook. The original dataset is embedded in the first code cell, so the notebook can run in Google Colab without uploading a separate CSV. Run cells from top to bottom.